# 03 · Scenario analysis

Sweep a model over many input sets and collect outputs into a tidy
DataFrame for sensitivity analysis.

In [ ]:
from dataclasses import dataclass

from finmodel import Model, row, Scenarios, PredefinedFormats as F



@dataclass

class Inputs:

    revenue: float

    growth_rate: float

    cost_ratio: float



class IncomeStatement(Model[Inputs]):

    @row(format=F.USD)

    def revenue(self, t):

        return self.inputs.revenue if t == 0 else self.revenue(t-1) * (1 + self.inputs.growth_rate)



    @row(format=F.USD)

    def ebitda(self, t):

        return self.revenue(t) * (1 - self.inputs.cost_ratio)



    @row(format=F.PERCENTAGE)

    def margin(self, t):

        return self.ebitda(t) / self.revenue(t)

## Register outputs and run a grid

`add_output(name, extractor)` registers a metric; the extractor receives
the freshly-calculated model. `run_scenarios` evaluates each input set.

In [ ]:
model = IncomeStatement(periods=5, inputs=Inputs(1_000, 0.10, 0.60))



scenarios = Scenarios(model)

scenarios.add_output("ebitda_y5", lambda m: m.get_result_data("ebitda")[-1])

scenarios.add_output("revenue_y5", lambda m: m.get_result_data("revenue")[-1])



grid = [

    Inputs(revenue=1_000, growth_rate=g, cost_ratio=c)

    for g in (0.05, 0.10, 0.15)

    for c in (0.55, 0.60)

]

scenarios.run_scenarios(grid)

df = scenarios.get_scenarios_df()

df

## Pivot into a sensitivity table

The columns are a two-level `('input'|'output', name)` index. Flatten
and pivot to get a classic growth × cost-ratio sensitivity grid.

In [ ]:
flat = scenarios.get_scenarios_df()

flat.columns = ["_".join(c) for c in flat.columns]

pivot = flat.pivot(index="input_growth_rate",

                   columns="input_cost_ratio",

                   values="output_ebitda_y5")

pivot